# Tutorial: Análise de Cadeias de Valor com Knowledge Graph Federado

**V-ANTPC** - Value Chain Analysis with Networked Triple-Pattern Cognition

Este notebook demonstra como usar o sistema de Knowledge Graph para análise de cadeias de valor organizacionais.

---

## 📚 Índice

1. Setup e Conexão com Fuseki
2. Carregamento de Ontologias
3. Análise Econômica (e3value + REA)
4. Análise de Capacidades (VDML)
5. Análise de Supply Chain (SCOR)
6. Visualização de Grafos
7. Casos de Uso Avançados

---

## 1️⃣ Setup e Conexão com Fuseki

### Pré-requisitos

Antes de executar este notebook, certifique-se de que:

1. **Docker está instalado e rodando**
2. **Fuseki foi iniciado** via `./setup.sh`
3. **Porta 3030 está disponível**

Se ainda não executou o setup, abra um terminal e execute:

```bash
cd v-antpc
./setup.sh
```

In [ ]:
# Importar bibliotecas necessárias
import sys
from pathlib import Path

# Adicionar diretório do projeto ao path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'knowledge_graph'))

from value_chain_federator import ValueChainFederator

print("✓ Bibliotecas importadas com sucesso")

In [ ]:
# Inicializar o federator
fed = ValueChainFederator(
    endpoint="http://localhost:3030/kg",
    user="admin",
    password="admin"
)

print("Federador inicializado.")
print(f"Endpoint: {fed.endpoint}")

In [ ]:
# Verificar conectividade com Fuseki
if fed.verificar_conexao():
    print("\n✅ Conexão estabelecida com sucesso!")
    print("Você pode acessar a interface web em: http://localhost:3030")
else:
    print("\n❌ Não foi possível conectar ao Fuseki.")
    print("Verifique se o Docker está rodando e se executou ./setup.sh")

## 2️⃣ Carregamento de Ontologias

O sistema usa **4 ontologias complementares**:

- **e3value**: Trocas de valor e reciprocidade econômica
- **REA**: Recursos, Eventos, Agentes (dualidade econômica)
- **VDML**: Capacidades organizacionais e propostas de valor
- **SCOR**: Processos de supply chain e métricas de desempenho

In [ ]:
# Carregar todas as ontologias
results = fed.carregar_todas_ontologias()

print("\n📊 Resumo do Carregamento:")
for ontologia, status in results.items():
    emoji = "✅" if status else "❌"
    print(f"{emoji} {ontologia}: {'Carregada' if status else 'Falhou'}")

In [ ]:
# Estatísticas gerais do grafo
stats = fed.estatisticas_grafo()

print("\n📈 Estatísticas do Knowledge Graph:")
print(f"Total de triplas: {stats.get('total_triplas', 0):,}")
print(f"Total de classes: {stats.get('total_classes', 0)}")
print(f"Total de atores: {stats.get('total_atores', 0)}")
print(f"Total de capacidades: {stats.get('total_capacidades', 0)}")

## 3️⃣ Análise Econômica (e3value + REA)

### 3.1 Listar Atores da Rede de Valor

In [ ]:
# Listar todos os atores
atores = fed.listar_atores()

print(f"\n🏢 Total de atores na rede: {len(atores)}\n")

if atores:
    print("Atores identificados:")
    for ator in atores[:10]:  # Mostrar primeiros 10
        nome = ator['uri'].split('/')[-1]
        tipo = ator['tipo'].split('#')[-1]
        print(f"  • {nome} ({tipo})")
else:
    print("⚠️ Nenhum ator encontrado. Você precisa carregar dados de exemplo.")
    print("Execute a Seção 7 para inserir dados do APL Têxtil de PE.")

### 3.2 Mapear Trocas de Valor

In [ ]:
# Listar trocas de valor entre atores
trocas = fed.listar_trocas_valor()

print(f"\n🔄 Total de trocas de valor: {len(trocas)}\n")

if trocas:
    print("Exemplos de trocas:")
    for troca in trocas[:5]:
        provedor = troca['provedor'].split('/')[-1]
        receptor = troca['receptor'].split('/')[-1]
        objeto = troca['valor_objeto'].split('/')[-1]
        print(f"  {provedor} → {receptor}: {objeto}")
else:
    print("⚠️ Nenhuma troca encontrada. Carregue dados de exemplo primeiro.")

### 3.3 Calcular Lucratividade de um Ator

In [ ]:
# Exemplo: calcular lucratividade de um ator específico
# Substitua pela URI real de um ator do seu grafo

exemplo_ator_uri = "http://apl-textil.pe/actor/RotaDoMar"

resultado = fed.calcular_lucratividade_ator(exemplo_ator_uri)

if resultado:
    print(f"\n💰 Análise de Lucratividade - {exemplo_ator_uri.split('/')[-1]}\n")
    print(f"Receita Total: R$ {resultado['receita']:,.2f}")
    print(f"Custo Total:   R$ {resultado['custo']:,.2f}")
    print(f"Lucro Líquido: R$ {resultado['lucro']:,.2f}")
    print(f"Margem:        {resultado['margem_percentual']:.2f}%")
else:
    print(f"⚠️ Não foi possível calcular lucratividade para {exemplo_ator_uri}")
    print("Verifique se o ator existe e tem dados econômicos associados.")

### 3.4 Verificar Dualidade Econômica (REA)

O modelo REA exige que **todo evento de incremento tenha um decremento dual** (e vice-versa).

In [ ]:
# Analisar pares de eventos duais
pares_duais = fed.analisar_dualidade_rea()

print(f"\n⚖️ Pares de Eventos Duais (REA): {len(pares_duais)}\n")

if pares_duais:
    for par in pares_duais[:3]:
        print(f"Incremento: {par['incremento'].split('/')[-1]}")
        print(f"  ↔ Decremento: {par['decremento'].split('/')[-1]}")
        print(f"Recurso: {par['recurso'].split('/')[-1]}")
        print(f"Quantidade: {par['quantidade']}")
        print("---")
else:
    print("⚠️ Nenhum par dual encontrado. Carregue dados de exemplo.")

## 4️⃣ Análise de Capacidades (VDML)

### 4.1 Listar Capacidades Organizacionais

In [ ]:
# Listar todas as capacidades
capacidades = fed.listar_capacidades()

print(f"\n🎯 Total de capacidades: {len(capacidades)}\n")

if capacidades:
    for cap in capacidades[:10]:
        nome = cap['uri'].split('/')[-1]
        is_core = "CORE" if cap['is_core'] == 'true' else "SUPPORT"
        nivel = cap.get('nivel', 'N/A')
        print(f"  [{is_core}] {nome} (Nível: {nivel}/5)")
else:
    print("⚠️ Nenhuma capacidade encontrada.")

### 4.2 Identificar Capacidades Core (Diferenciadoras)

In [ ]:
# Filtrar apenas capacidades core
capacidades_core = fed.listar_capacidades(apenas_core=True)

print(f"\n🌟 Capacidades CORE (Diferenciadoras): {len(capacidades_core)}\n")

if capacidades_core:
    for cap in capacidades_core:
        print(f"✓ {cap['descricao']}")
        print(f"  Nível de Maturidade: {cap.get('nivel', 'N/A')}/5\n")
else:
    print("⚠️ Nenhuma capacidade core encontrada.")

## 5️⃣ Análise de Supply Chain (SCOR)

### 5.1 Listar Métricas SCOR

In [ ]:
# Listar métricas de desempenho SCOR
metricas = fed.listar_metricas_scor()

print(f"\n📊 Métricas SCOR: {len(metricas)}\n")

if metricas:
    print(f"{'Métrica':<40} {'Atual':<15} {'Alvo':<15} {'Gap':<10}")
    print("-" * 80)
    
    for metrica in metricas[:10]:
        nome = metrica['metrica'].split('/')[-1]
        atual = metrica.get('valor_atual', 'N/A')
        alvo = metrica.get('valor_alvo', 'N/A')
        unidade = metrica.get('unidade', '')
        
        try:
            gap = float(alvo) - float(atual)
            print(f"{nome:<40} {atual} {unidade:<10} {alvo} {unidade:<10} {gap:.2f}")
        except (ValueError, TypeError):
            print(f"{nome:<40} {atual} {unidade:<10} {alvo} {unidade:<10} -")
else:
    print("⚠️ Nenhuma métrica SCOR encontrada.")

## 6️⃣ Consultas SPARQL Customizadas

### 6.1 Executar Query Personalizada

In [ ]:
# Exemplo de query SPARQL customizada
query = """
SELECT ?ator ?nome (COUNT(?interface) AS ?numInterfaces)
WHERE {
    ?ator a e3:Actor .
    OPTIONAL { ?ator rdfs:label ?nome }
    OPTIONAL { ?ator e3:has_value_interface ?interface }
}
GROUP BY ?ator ?nome
ORDER BY DESC(?numInterfaces)
LIMIT 10
"""

results = fed.consultar(query)

if results and 'results' in results:
    print("\n🔍 Atores por número de interfaces de valor:\n")
    
    for binding in results['results']['bindings']:
        ator = binding['ator']['value'].split('/')[-1]
        nome = binding.get('nome', {}).get('value', 'N/A')
        num_interfaces = binding['numInterfaces']['value']
        print(f"  {ator}: {num_interfaces} interfaces")
else:
    print("⚠️ Query não retornou resultados")

### 6.2 Carregar e Executar Query de Arquivo

In [ ]:
# Ler query de arquivo SPARQL
query_file = project_root / 'knowledge_graph' / 'sparql_queries' / 'analise_economica.sparql'

if query_file.exists():
    with open(query_file, 'r', encoding='utf-8') as f:
        content = f.read()
        
        # Pegar primeira query (até o segundo #===)
        queries = content.split('# ==============')
        if len(queries) > 2:
            first_query = queries[2].split('SELECT')[1].split('\n\n')[0]
            first_query = 'SELECT' + first_query
            
            print("📄 Executando query do arquivo analise_economica.sparql...\n")
            print(f"Query: {first_query[:100]}...\n")
            
            results = fed.consultar(first_query)
            
            if results and 'results' in results:
                print(f"✓ Retornou {len(results['results']['bindings'])} resultados")
            else:
                print("Query executada, mas sem resultados")
else:
    print(f"⚠️ Arquivo não encontrado: {query_file}")

## 7️⃣ Inserir Dados de Exemplo (APL Têxtil de PE)

### 7.1 Inserir Ator de Exemplo

In [ ]:
# Exemplo de inserção de triplas RDF
triplas_exemplo = """
@prefix e3: <http://e3value.com/ontology#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ex: <http://apl-textil.pe/> .

ex:actor/RotaDoMar a e3:Actor ;
    rdfs:label "Rota do Mar - Confecções"@pt ;
    rdfs:comment "Empresa de referência do APL Têxtil de Pernambuco, fundada em 1996"@pt ;
    e3:profitability 50.3 .

ex:actor/PoloToritamaJeans a e3:MarketSegment ;
    rdfs:label "Polo Toritama Jeans"@pt ;
    rdfs:comment "Agregação de 3000+ empresas especializadas em jeans"@pt .
"""

sucesso = fed.inserir_triplas(
    triplas_exemplo,
    graph_uri="http://valuechain.org/data/apl-textil-pe"
)

if sucesso:
    print("✅ Dados de exemplo inseridos com sucesso!")
    print("\nExecute novamente as células de análise para ver os resultados.")
else:
    print("❌ Erro ao inserir dados")

### 7.2 Verificar Inserção

In [ ]:
# Re-listar atores para verificar inserção
atores_atualizados = fed.listar_atores()

print(f"\n🏢 Atores após inserção: {len(atores_atualizados)}")

if atores_atualizados:
    for ator in atores_atualizados:
        nome = ator['uri'].split('/')[-1]
        print(f"  • {nome}")

## 8️⃣ Limpeza e Reset

### 8.1 Limpar Dados de Instância (Manter Ontologias)

In [ ]:
# CUIDADO: Isso vai remover todos os dados de instância!
# Descomente a linha abaixo apenas se tiver certeza

# fed.limpar_grafo(graph_uri="http://valuechain.org/data/apl-textil-pe")

print("⚠️ Limpeza desabilitada por segurança.")
print("Descomente o código acima para executar.")

## 🎓 Conclusão

Você aprendeu a:

✅ Conectar ao Fuseki e carregar ontologias  
✅ Listar atores, trocas e capacidades  
✅ Calcular lucratividade de atores  
✅ Verificar dualidade econômica (REA)  
✅ Analisar capacidades core vs support  
✅ Executar queries SPARQL customizadas  
✅ Inserir dados de exemplo  

---

### 📚 Próximos Passos

1. **Carregar datasets completos**: Use os arquivos em `data/apl_textil_pe_parte*.md`
2. **Explorar queries avançadas**: Veja `knowledge_graph/sparql_queries/`
3. **Visualizar grafos**: Implemente visualizações com PyVis
4. **Criar análises customizadas**: Combine múltiplas ontologias

---

**Documentação**: Veja `README.md` e `QUICKSTART.md` para mais detalhes.
